# 🤖 IA — Predicción de Revenue (Forecasting)

Este notebook implementa **forecasting con Machine Learning** sobre los datos
de reservas confirmadas, prediciendo el revenue de los próximos 6 meses.

## Cómo funciona

1. Lee las reservas confirmadas de la capa Gold (`gold.gold_fact_reservas`).
2. Agrupa por mes y calcula el revenue histórico.
3. Entrena un modelo **Prophet** (algoritmo open source de Facebook/Meta).
4. Predice los próximos 6 meses con intervalos de confianza.
5. Guarda el resultado en `gold.gold_revenue_forecast` listo para Power BI.

## Por qué Prophet

- Maneja **estacionalidad** automáticamente (detecta picos como julio).
- Robusto a **datos faltantes** o outliers.
- Solo requiere 2 columnas: fecha y valor.
- Estándar de la industria (Facebook, Uber, Airbnb lo usan internamente).
- Implementación de 4 líneas de código.

## Para la demo

1. **Run all** en este notebook (1 minuto).
2. En Power BI, agregar el gráfico de línea con los datos reales + predichos.
3. Refrescar el dashboard para mostrar el forecast en vivo.

## 1. Instalar Prophet (solo la primera vez)

In [ ]:
%pip install prophet --quiet

In [ ]:
dbutils.library.restartPython()

## 2. Cargar datos históricos de Gold

Agrupa las reservas confirmadas por mes para obtener la serie temporal.

In [ ]:
# Leer revenue mensual histórico (solo reservas confirmadas)
df_historico = spark.sql("""
  SELECT
    DATE_TRUNC('month', tiempo_id) AS ds,
    SUM(total_amount)              AS y
  FROM gold.gold_fact_reservas
  WHERE booking_status = 'confirmed'
  GROUP BY DATE_TRUNC('month', tiempo_id)
  ORDER BY ds
""").toPandas()

print(f"Histórico cargado: {len(df_historico)} meses")
print(f"Rango: {df_historico['ds'].min()} → {df_historico['ds'].max()}")
print(f"Revenue total histórico: ${df_historico['y'].sum():,.0f}")
df_historico.head()

## 3. Entrenar el modelo Prophet

Prophet entrena en segundos y detecta estacionalidad automáticamente.

In [ ]:
from prophet import Prophet

# Configurar modelo con estacionalidad anual (turismo es estacional)
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.80  # Intervalo de confianza del 80%
)

# Entrenar
modelo.fit(df_historico)

print("✓ Modelo Prophet entrenado correctamente")

## 4. Predecir los próximos 6 meses

In [ ]:
# Generar fechas futuras (6 meses)
futuro = modelo.make_future_dataframe(periods=6, freq='MS')

# Predecir
prediccion = modelo.predict(futuro)

# Quedarnos solo con las columnas relevantes
resultado = prediccion[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
resultado.columns = ['fecha', 'revenue_predicho', 'limite_inferior', 'limite_superior']

# Marcar qué es histórico vs predicción
import pandas as pd
fechas_historicas = set(df_historico['ds'].dt.to_pydatetime())
resultado['tipo'] = resultado['fecha'].apply(
    lambda x: 'histórico' if x in fechas_historicas else 'predicción'
)

print(f"Total filas (histórico + 6 meses futuros): {len(resultado)}")
resultado.tail(10)

## 5. Guardar predicción en la capa Gold

La tabla queda lista para que Power BI la consuma vía DirectQuery.

In [ ]:
# Convertir a Spark DataFrame y guardar como tabla Delta
(
    spark.createDataFrame(resultado)
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable("gold.gold_revenue_forecast")
)

print("✓ Predicción guardada en gold.gold_revenue_forecast")

## 6. Validación — ver la predicción

In [ ]:
%sql
-- Ver los últimos 6 meses históricos + los 6 meses predichos
SELECT
  fecha,
  ROUND(revenue_predicho, 0)  AS revenue_predicho_usd,
  ROUND(limite_inferior, 0)   AS limite_inferior_usd,
  ROUND(limite_superior, 0)   AS limite_superior_usd,
  tipo
FROM gold.gold_revenue_forecast
ORDER BY fecha DESC
LIMIT 12;

## 7. KPI: revenue esperado próximos 6 meses

In [ ]:
%sql
SELECT
  ROUND(SUM(revenue_predicho), 0)  AS revenue_esperado_6m,
  ROUND(SUM(limite_inferior), 0)   AS escenario_pesimista,
  ROUND(SUM(limite_superior), 0)   AS escenario_optimista
FROM gold.gold_revenue_forecast
WHERE tipo = 'predicción';

## 🎬 Próximo paso: visualizar en Power BI

### Opción A — Forecast nativo de Power BI (1 clic)

En tu Dashboard 1, en el gráfico **"INGRESOS POR MES"**:

1. Clic en el gráfico de línea.
2. Panel **Análisis** (icono de lupa con `f(x)`).
3. Sección **"Pronóstico"** → **Agregar**.
4. Configurar:
   - **Longitud del pronóstico**: 6 meses
   - **Intervalo de confianza**: 80%
5. Aplicar.

Power BI dibuja una línea sombreada con la proyección. Es ARIMA simple integrado.

### Opción B — Tabla `gold_revenue_forecast` cargada

Usa esta tabla directamente para crear visuales más sofisticados:

1. En Power BI: **Inicio** → **Actualizar** → cargar `gold.gold_revenue_forecast`.
2. Crear visual de línea:
   - **Eje X**: `fecha`
   - **Eje Y**: `revenue_predicho`
   - **Leyenda**: `tipo` (histórico vs predicción)
3. Agregar **`limite_superior`** y **`limite_inferior`** como bandas de confianza.
4. Título: `REVENUE FORECAST — PRÓXIMOS 6 MESES`

Resultado: línea con sombra mostrando la incertidumbre de la predicción.

## Defensa para la sustentación

> *"Implementamos forecasting de revenue con **Prophet**, el modelo open source de Facebook/Meta que detecta estacionalidad automáticamente. Entrenamos sobre los datos históricos de reservas confirmadas de la capa Gold, generando predicciones de los próximos 6 meses con intervalos de confianza del 80%. El resultado se persiste en `gold.gold_revenue_forecast` y se consume desde Power BI vía DirectQuery — permitiendo que el equipo comercial planifique capacidad e inversión con base en proyecciones de revenue confiables. Es ML aplicado al negocio: pasamos de descriptive analytics (qué pasó) a predictive analytics (qué pasará)."*

## Decisiones de diseño

- **¿Por qué Prophet y no AutoML?** Prophet requiere 5 líneas de código vs AutoML que necesita configuración de cluster ML. Para series temporales con estacionalidad clara (turismo), Prophet supera a AutoML en simplicidad y robustez.
- **¿Por qué solo reservas confirmadas?** Para predecir revenue real (no GMV). El modelo aprende del dinero que efectivamente entró.
- **¿Por qué intervalo del 80%?** Más estricto que el default (95%) — comunica incertidumbre realista al equipo comercial.
- **¿Por qué guardar en Delta?** Para que Power BI consuma vía DirectQuery sin recalcular. El forecast se regenera mensualmente con un Databricks Job.